# β-VAE exploration

The loss is the per-image sum of squared error plus β times the closed-form KL. β = 1 is the standard VAE. The latent width is fixed at 8. Stage 07 fits β ∈ {0.1, 1, 4, 10} from the same initialization and writes `reports/mnist_betta_vae/beta_exploration.json`.

Read the three effects together:

- **Reconstruction.** Test MSE, posterior-mean decode. Smaller β can copy the pixels more closely. Larger β blurs them.
- **Disentanglement.** `active_units` counts latent axes whose posterior mean still varies across the test images (variance above 0.01). That is a compactness proxy. MNIST does not label stroke width or slant, so the traversal grids are the check: one row is one axis of the zero code, walked from −3 to 3. `traversal_response` is the mean pixel variance along that row.
- **Generation diversity.** Mean pairwise per-pixel squared distance among 64 prior samples. A collapsed generator scores lower.

`reports/mnist_betta_vae/plots/loss_vs_beta.png` is the β-weighted validation loss. A smaller β lowers that number by shrinking the KL term. It is not the disentanglement ranking. Run `make all` first if the report below is missing.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
for _candidate in (ROOT, *ROOT.parents):
    if (_candidate / "pyproject.toml").is_file():
        ROOT = _candidate
        break
report_path = ROOT / "reports" / "mnist_betta_vae" / "beta_exploration.json"
if not report_path.is_file():
    raise FileNotFoundError(f"missing {report_path}. Run make all from the repository root.")

report = json.loads(report_path.read_text(encoding="utf-8"))
frame = pd.DataFrame(report["runs"])[
    ["beta", "mse", "mae", "kl", "elbo", "active_units", "diversity"]
]
frame


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(9.6, 3.2))
series = (
    ("mse", "reconstruction MSE"),
    ("active_units", "active latent units"),
    ("diversity", "generation diversity"),
)
for axis, (key, title) in zip(axes, series, strict=True):
    axis.plot(frame["beta"], frame[key], marker="o")
    axis.set_xlabel("beta")
    axis.set_title(title)
    axis.set_xticks(list(frame["beta"]))
fig.tight_layout()
plt.show()


In [ ]:
from IPython.display import Image, display

plots = ROOT / "reports" / "mnist_betta_vae" / "plots"
for row in report["runs"]:
    path = plots / row["traversal"]
    print(f"beta={row['beta']:g}  traversal_response={['%.4f' % value for value in row['traversal_response']]}")
    display(Image(filename=str(path)))


## How to read a finished run

A higher β should raise test MSE, turn some latent rows flat (`traversal_response` near 0, fewer `active_units`), and shrink generation diversity when the prior samples start to look alike. If two rows of one grid change the same stroke, those axes are still entangled. Compare a row at β = 0.1 with the same row index at β = 10: the low-β walk usually changes several digit properties at once, and the high-β walk either changes one property or barely changes the image.
